# Problem Set 8

[PSet 8](pset8.pdf)

In [1]:
import sympy as sp
import sympy.physics.mechanics as spm
import sympy.physics.vector as spv
from sympy.physics.vector.printing import init_vprinting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import IPython

# Import IPython display for proper LaTeX formatting
from IPython.display import display, Math, Markdown

# Initialize symbols
sp.init_printing()

# Enable dot notation printing for dynamicsymbols
init_vprinting(use_latex="mathjax")

HALF = sp.S.Half

In [2]:
def reference_frame(frame: str, x=r"\imath", y=r"\jmath", z=r"k") -> spm.ReferenceFrame:
    """Create a SymPy reference frame with custom basis vector labels.

    Parameters
    ----------
    frame : str
        The name of the reference frame.
    x, y, z : str
        Labels for the basis vectors.
    """
    return spm.ReferenceFrame(
        frame,
        latexs=(
            rf"\;{{}}^\mathcal{{{frame}}}\hat{{{x}}}",
            rf"\;{{}}^\mathcal{{{frame}}}\hat{{{y}}}",
            rf"\;{{}}^\mathcal{{{frame}}}\hat{{{z}}}",
        ),
    )


def reference_frame_circular(name: str, angle=r"theta") -> spm.ReferenceFrame:
    """Create a circular reference frame with radial and angular basis labels.

    Parameters
    ----------
    name : str
        Name of the new reference frame.
    angle : str, optional
        Symbol or label used for the angular basis vector, by default "theta".
    """
    return reference_frame(name, x=r"r", y=rf"\{angle}", z=r"e_z")

## Problem 8.1 

Spring-Loop-the-Loop
A small block of mass m is pushed against a spring with spring constant k and held in
place with a catch. The spring compresses an unknown distance x. When the catch
is removed, the block leaves the spring and slides along a frictionless circular loop of
radius R


![Slide Down an Inclined Plane](../figures/PS0801-spring-block-loop.jpg)

When the block reaches the top of the loop, the force of the loop on the block (the
normal force) is equal to twice the gravitational force on the mass. How far was the
spring initially compressed? Write your answer using some or all of the following: g,
k, R, and m.

In [3]:
(
    m,  # mass of the object
    ell_0,  # length of the inclined plane
    k, # spring constant
    x, # compression of the spring
    g,  # acceleration due to gravity
    R  # distance it takes the object to stop measured from the bottom of the incline
) = sp.symbols("m ell_0 k x g R", real=True, positive=True)

theta = spm.dynamicsymbols("theta")

In [4]:
N = reference_frame("N")
M = reference_frame("M", x=r"r", y=r"\theta", z=r"e_z")
M.orient_axis(N, N.z, theta)   

vec_R = R * M.x
vec_v = vec_R.dt(N)
vec_a = vec_v.dt(N)

display(Math(rf"\text{{Position vector: }} {spv.vlatex(vec_R)}"))
display(Math(rf"\text{{Velocity vector: }} {spv.vlatex(vec_v)}"))
display(Math(rf"\text{{Acceleration vector: }} {spv.vlatex(vec_a)}")) 

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [5]:
# Define CME for top of loop
vf = sp.symbols("v_f", real=True, positive=True) # Velocity at the top of the loop

# Conservation of mechanical energy(CME) for spring at moment of release
CMEeqn = sp.Eq(sp.S.Half * m * vf**2 + m * g * 2 * R, sp.S.Half * k * x**2 )
vfsq_sol = sp.solve(CMEeqn, vf**2)[0]
display(Math(rf"\text{{Speed squared at the top of the loop: }} {spv.vlatex(vfsq_sol)}"))

<IPython.core.display.Math object>

In [6]:
# Define the forces acting on the object in circular loop
F_gravity = m * g * N.x
Normal_mag = 2 * m * g  # Normal force magnitude
F_Normal = Normal_mag * (-M.x)
F_total = F_gravity + F_Normal

# WET for the object at an angle pi in the circular loop.
# Top of the loop is theta=pi, bottom of the loop is theta=0

# N2L
N2Leqn = spm.msubs(
    sp.Eq(F_total.to_matrix(N), (m * vec_a.to_matrix(N))), {theta: sp.pi}
)
N2Leqn_solved = sp.solve(N2Leqn, [theta.diff() ** 2, theta.diff().diff()], dict=True)[0]

# Velocity squared at the top of the loop
x_solution = sp.solve(
    sp.Eq(R**2 * N2Leqn_solved[theta.diff() ** 2], vfsq_sol), x**2, dict=True
)[0]
x_solution_sqrt = sp.sqrt(x_solution[x**2].simplify(), evaluate=False)
display(
    Math(rf"\text{{Spring compression }} x: \boxed{{{spv.vlatex(x_solution_sqrt)}}}")
)

<IPython.core.display.Math object>